<a href="https://colab.research.google.com/github/chuckwang123/ConfigurationManager/blob/master/sequence_packing_cpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch;
from transformers import AutoTokenizer

In [ ]:
sentence1 = "green glass bottle broke inside that tiny wooden box today"
sentence2 = "red sudden butterfly"
sentence3 = "silver winter forest morning"

oneLineSentence = [sentence1, sentence2, sentence3]

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
inputs = tokenizer(oneLineSentence)
input_ids = inputs["input_ids"]
print("Token IDs:", input_ids)
print(len(inputs))


all_tokens = [tokenizer.convert_ids_to_tokens(ids) for ids in input_ids]

for i, tokens in enumerate(all_tokens):
    print(f"Sentence {i+1} Tokens:", tokens)

Token IDs: [[13250, 8991, 16486, 14422, 4766, 429, 13673, 22360, 3745, 3351], [1151, 10968, 55169], [81914, 12406, 13638, 6556]]
2
Sentence 1 Tokens: ['green', 'Ġglass', 'Ġbottle', 'Ġbroke', 'Ġinside', 'Ġthat', 'Ġtiny', 'Ġwooden', 'Ġbox', 'Ġtoday']
Sentence 2 Tokens: ['red', 'Ġsudden', 'Ġbutterfly']
Sentence 3 Tokens: ['silver', 'Ġwinter', 'Ġforest', 'Ġmorning']


In [ ]:
eos_id = tokenizer.encode("<|im_end|>", add_special_tokens=False)
combined_ids = []
for i, single_input_ids in enumerate(input_ids):
    combined_ids = combined_ids + single_input_ids + eos_id

tokens = tokenizer.convert_ids_to_tokens(combined_ids)
print("Tokens:", tokens)

Tokens: ['green', 'Ġglass', 'Ġbottle', 'Ġbroke', 'Ġinside', 'Ġthat', 'Ġtiny', 'Ġwooden', 'Ġbox', 'Ġtoday', '<|im_end|>', 'red', 'Ġsudden', 'Ġbutterfly', '<|im_end|>', 'silver', 'Ġwinter', 'Ġforest', 'Ġmorning', '<|im_end|>']


In [ ]:
input_ids_tensor = torch.tensor([combined_ids])
seq_len = input_ids_tensor.size(1)

tril_mask = torch.ones(seq_len, seq_len, dtype=torch.int).tril()

In [ ]:
tril_mask

tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1,

In [ ]:
block_mask = torch.zeros(seq_len, seq_len, dtype=torch.int)
lengths = [len(ids) + 1 for ids in input_ids]
pos_ids_list = []

current_idx = 0
for l in lengths:
    sub_mask = torch.ones(l, l, dtype=torch.int).tril()
    block_mask[current_idx:current_idx+l, current_idx:current_idx+l] = sub_mask
    current_idx += l

    sub_pos_ids = torch.arange(l, dtype=torch.int)
    pos_ids_list.append(sub_pos_ids)

position_ids = torch.cat(pos_ids_list)

print(block_mask.shape)
print("\n")
print(block_mask)
print("\n")
print(position_ids)

torch.Size([20, 20])


tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0